# s25 — Worktree Merge-Conflict Reporting

**What this teaches:** how the `worktree.Merge` helper reports conflicts when a merge fails. Where [s24_worktree](../s24_worktree/s24_worktree.ipynb) only describes the tools, this notebook *creates* two branches with overlapping edits and watches the second merge abort with a conflict report.

**Concept anchor:** the agent should never be allowed to silently force-merge over conflicts. `worktree.Merge` returns a structured report containing the offending files so the host (or the leader) can surface them to a human.

## Prerequisites

- `git` available on `$PATH`.
- No LLM is required for this demo — it exercises `internal/worktree` directly. Add an agent flow if you want the model to *explain* the conflict report.
- A temporary directory is created in `/tmp` and cleaned up at the end.

## 1. Imports

We use `os/exec` to script the git setup so the conflict scenario is reproducible.

In [ ]:
import (
    "fmt"
    "os"
    "os/exec"

    wt "github.com/blouargant/yoke/internal/worktree"
)

## 2. Helpers

`must` panics on errors (notebook-safe); `run` shells out and streams output.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

func run(name string, args ...string) {
    c := exec.Command(name, args...)
    c.Stdout, c.Stderr = os.Stdout, os.Stderr
    must(c.Run())
}

## 3. Set up a throwaway repository

We create a fresh repo in a temp dir — touching the surrounding Yoke checkout would be rude, and we want full control over branch history.

In [ ]:
tmp, err := os.MkdirTemp("", "s25-conflict-")
must(err)
defer os.RemoveAll(tmp)
must(os.Chdir(tmp))

run("git", "init", "-q", "-b", "main")
run("git", "config", "user.email", "demo@example.com")
run("git", "config", "user.name", "demo")
must(os.WriteFile("file.txt", []byte("base\n"), 0o644))
run("git", "add", ".")
run("git", "commit", "-q", "-m", "base")
fmt.Println("base commit ready in", tmp)

## 4. Create two conflicting branches

Both branches modify the same line of `file.txt` — classic merge-conflict setup.

In [ ]:
run("git", "checkout", "-q", "-b", "feat-a")
must(os.WriteFile("file.txt", []byte("from A\n"), 0o644))
run("git", "commit", "-q", "-am", "A change")

run("git", "checkout", "-q", "main")
run("git", "checkout", "-q", "-b", "feat-b")
must(os.WriteFile("file.txt", []byte("from B\n"), 0o644))
run("git", "commit", "-q", "-am", "B change")

run("git", "checkout", "-q", "main")
fmt.Println("two branches set up: feat-a, feat-b")

## 5. Merge `feat-a` cleanly

The first merge fast-forwards or commits cleanly. `wt.Merge` returns a textual report.

In [ ]:
report, err := wt.Merge(tmp, "feat-a")
must(err)
fmt.Println("── feat-a merge report ──")
fmt.Println(report)

## 6. Merge `feat-b` → expect conflict

Now the second merge hits the same line `feat-a` already changed. `wt.Merge` aborts the merge and returns an error *plus* a report listing the conflicting files.

In [ ]:
report, err = wt.Merge(tmp, "feat-b")
if err != nil {
    fmt.Println("conflict path triggered as expected:", err)
}
fmt.Println("── feat-b merge report ──")
fmt.Println(report)

## What to look for

- The first report says success; the second says conflict and lists `file.txt`.
- The repo state after the failed merge is **clean** — `wt.Merge` aborts on conflict, it does not leave a half-merged tree. That's the contract that lets the leader recover safely.
- Compare to [s24_worktree](../s24_worktree/s24_worktree.ipynb): that notebook only *describes* the tools; this one *runs* the most interesting failure path.

## Try it yourself

1. After the conflict, manually resolve `file.txt` and re-run a `git merge feat-b` from a terminal — observe `wt.Merge` is not needed once the human is in the loop.
2. Wrap this in an agent (give it `bash` + the worktree tools) and ask it: "merge feat-a then feat-b and explain any conflict" — the model now becomes the human in step 1.